# Finetuning Mask R-CNN

based on this tutorial
https://pytorch.org/tutorials/intermediate/torchvision_tutorial.html


In [ ]:
import matplotlib.pyplot as plt
from torchvision.io import read_image

image = read_image(
    "../../training_data/images/A-6_default_mmrsbpb_mMRS-BPB_15122023130354.jpg"
)
mask = read_image(
    "../../training_data/masks/A-6_default_mmrsbpb_mMRS-BPB_15122023130354.png"
)

plt.figure(figsize=(16, 8))
plt.subplot(121)
plt.title("Image")
plt.imshow(image.permute(1, 2, 0))
plt.subplot(122)
plt.title("Mask")
plt.imshow(mask.permute(1, 2, 0))

In [ ]:
import os

import torch
from torchvision import tv_tensors
from torchvision.io import read_image
from torchvision.ops.boxes import masks_to_boxes
from torchvision.transforms.v2 import functional as F


class ColonySegmentationDataset(torch.utils.data.Dataset):
    def __init__(self, root, transforms):
        self.root = root
        self.transforms = transforms
        # load all image files, sorting them to ensure that they are aligned
        self.imgs = list(sorted(os.listdir(os.path.join(root, "images"))))
        self.masks = list(sorted(os.listdir(os.path.join(root, "masks"))))

    def __getitem__(self, idx):
        img_path = os.path.join(self.root, "images", self.imgs[idx])
        mask_path = os.path.join(self.root, "masks", self.masks[idx])
        img = read_image(img_path)
        mask = read_image(mask_path)

        # turn mask from [C, H, W] to [H, W] by combining the color channels into a single integer
        int_mask = (mask[0].int() << 16) + (mask[1].int() << 8) + mask[2].int()
        # Find unique integer values (now each integer represents a unique color)
        obj_ids = torch.unique(int_mask)
        # first id is the background (black), so remove it
        obj_ids = obj_ids[1:]
        num_objs = len(obj_ids)

        # split the color-encoded mask into a set of binary masks
        masks = (int_mask == obj_ids[:, None, None]).to(dtype=torch.uint8)

        boxes = masks_to_boxes(masks)

        # there is only one class (colony)
        labels = torch.ones((num_objs,), dtype=torch.int64)

        image_id = idx
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        # suppose all instances are not crowd
        iscrowd = torch.zeros((num_objs,), dtype=torch.int64)

        # Wrap sample and targets into torchvision tv_tensors for transforms
        img = tv_tensors.Image(img)

        target = {}
        target["boxes"] = tv_tensors.BoundingBoxes(
            boxes, format="XYXY", canvas_size=F.get_size(img)
        )
        target["masks"] = tv_tensors.Mask(masks)
        target["labels"] = labels
        target["image_id"] = image_id
        target["area"] = area
        target["iscrowd"] = iscrowd

        if self.transforms is not None:
            img, target = self.transforms(img, target)

        return img, target

    def __len__(self):
        return len(self.imgs)

In [ ]:
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor


def get_model_instance_segmentation(num_classes):
    # load an instance segmentation model pre-trained on COCO
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights="DEFAULT")

    # get number of input features for the classifier
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    # replace the pre-trained head with a new one
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # now get the number of input features for the mask classifier
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    # and replace the mask predictor with a new one
    model.roi_heads.mask_predictor = MaskRCNNPredictor(
        in_features_mask, hidden_layer, num_classes
    )

    return model

In [ ]:
from torchvision.transforms import v2 as T


def get_transforms(train):
    transforms = []
    if train:
        transforms.append(T.RandomHorizontalFlip(0.5))
        transforms.append(T.RandomVerticalFlip(0.5))
        # Rotation by 0, 90, 180, or 270 degrees
        transforms.append(
            T.RandomChoice(
                [
                    T.RandomRotation((0, 0)),
                    T.RandomRotation((90, 90)),
                    T.RandomRotation((180, 180)),
                    T.RandomRotation((270, 270)),
                ]
            )
        )
        transforms.append(T.ColorJitter(brightness=0.1, contrast=0.1))
    transforms.append(T.ToDtype(torch.float, scale=True))
    transforms.append(T.ToPureTensor())
    return T.Compose(transforms)

# Testing `forward()` method

Before iterating over the dataset, it's good to see what the model expects
during training and inference time on sample data.


In [ ]:
import utils

model = get_model_instance_segmentation(2)
dataset = ColonySegmentationDataset("../../training_data", get_transforms(train=True))
data_loader = torch.utils.data.DataLoader(
    dataset, batch_size=2, shuffle=True, collate_fn=utils.collate_fn
)

# For Training
images, targets = next(iter(data_loader))
images = list(image for image in images)
targets = [{k: v for k, v in t.items()} for t in targets]
output = model(images, targets)  # Returns losses and detections
print(output)

# For inference
model.eval()
x = [torch.rand(3, 300, 400), torch.rand(3, 500, 400)]
predictions = model(x)  # Returns predictions
print(predictions[0])

## Training and Validation


In [ ]:
# train on the GPU or on the CPU, if a GPU is not available
device = torch.device("cpu")

# Check for accelerator availability
# if torch.backends.mps.is_available():
#     device = torch.device("mps")
#     print("Using MPS for training.")
# elif torch.cuda.is_available():
#     device = torch.device("cuda")
#     print("Using CUDA for training.")
# else:
#     print("Using CPU for training.")

In [ ]:
from engine import evaluate, train_one_epoch


# our dataset has two classes only - background and colony
num_classes = 2
# use our dataset and defined transformations
dataset = ColonySegmentationDataset("../../training_data", get_transforms(train=True))
dataset_test = ColonySegmentationDataset(
    "../../training_data", get_transforms(train=False)
)

# split the dataset in train and test set (last 30 images are test set)
indices = torch.randperm(len(dataset)).tolist()
dataset = torch.utils.data.Subset(dataset, indices[:-30])
dataset_test = torch.utils.data.Subset(dataset_test, indices[-30:])

# define training and validation data loaders
data_loader = torch.utils.data.DataLoader(
    dataset, batch_size=2, shuffle=True, collate_fn=utils.collate_fn
)

data_loader_test = torch.utils.data.DataLoader(
    dataset_test,
    batch_size=1,
    shuffle=False,
    collate_fn=utils.collate_fn,
)

model = get_model_instance_segmentation(num_classes)
model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

num_epochs = 2

for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq=10)
    # update the learning rate
    lr_scheduler.step()
    # evaluate on the test dataset
    evaluate(model, data_loader_test, device=device)

print("That's it!")

# Save the model to disk
torch.save(model.state_dict(), "model.pth")

# Gradio app for trying it out


In [ ]:
import gradio as gr
import matplotlib.pyplot as plt
from torchvision.utils import draw_bounding_boxes, draw_segmentation_masks

model = get_model_instance_segmentation(2)
model.load_state_dict(torch.load("model.pth"))

eval_transform = get_transforms(train=False)


def predict(image):
    # convert numpy image to tensor
    image = torch.tensor(image).permute(2, 0, 1).float() / 255.0

    model.eval()
    with torch.no_grad():
        print(type(image))
        x = eval_transform(image)
        print(type(x))
        # convert RGBA -> RGB and move to device
        x = x[:3, ...].to(device)
        predictions = model([x])
        pred = predictions[0]

    image = (255.0 * (image - image.min()) / (image.max() - image.min())).to(
        torch.uint8
    )
    image = image[:3, ...]
    pred_labels = [
        f"colony: {score:.3f}" for label, score in zip(pred["labels"], pred["scores"])
    ]
    pred_boxes = pred["boxes"].long()
    output_image = draw_bounding_boxes(
        image, pred_boxes, pred_labels, colors="red", width=5, font_size=1
    )

    masks = (pred["masks"] > 0.7).squeeze(1)
    output_image = draw_segmentation_masks(
        output_image, masks, alpha=0.5, colors="yellow"
    )

    return output_image.permute(1, 2, 0).numpy()


demo = gr.Interface(
    fn=predict,
    inputs="image",
    outputs="image",
    title="Colony Detection",
    description="Detect colonies in images.",
)

demo.launch(inline=False)